In [11]:
import json
import shutil
from pathlib import Path
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import uuid
from dotenv import load_dotenv

import os
load_dotenv()

# Set up OpenAI API key
openai_api_key = os.getenv("OPENAI_API_KEY")
# client= OpenAI()
class VideoProcessorQdrant:
    def __init__(
        self,
        collection_name,
        embedding_model="text-embedding-3-large",
        clear_storage=False,
        use_scene_description=False,
        openai_api_key=openai_api_key,
        region = None,
        file_path=None,
        reset_collection=False,
    ):
        """
        Initialize the VideoProcessorQdrant.

        Args:
            collection_name (str): Name of the Qdrant collection.
            storage_path (str): Path to the persistent storage.
            embedding_model (str): Name of the embedding model.
            clear_storage (bool): Whether to clear existing storage at initialization.
            use_scene_description (bool): Whether to use scene descriptions.
        """
        self.collection_name = collection_name
        self.embedding_model = embedding_model
        self.openai_client = OpenAI(api_key=openai_api_key)
        self.file_path = file_path
        self.reset_collection = reset_collection
        self.region = region


        
        # Initialize Qdrant client
        # self.client = QdrantClient(url="http://localhost", port=6333)
        self.client = QdrantClient(url="http://dev.platform.farmer.chat:5438/", port=5438, grpc_port=5439, prefer_grpc=False)

        # Check if the collection exists
        try:
            self.client.get_collection(collection_name)
            print(f"Using existing collection '{collection_name}'.")
        except Exception:  # Collection does not exist
            print(f"Creating collection '{collection_name}'...")
            self.client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
            )

    def get_embedding(self, text):
        """Get embedding for text using OpenAI."""
        response = self.openai_client.embeddings.create(
            input=text,
            model=self.embedding_model
        )
        return response.data[0].embedding

    def push_transcripts(self):
        """Push transcript data to Qdrant."""
        with open(self.file_path, "r") as f:
            data = json.load(f)
            chunks = data['chunks']

        if not chunks:
            print(f"chunks in this {file_path} is empty or invalid.")
            return
        points = []
        for doc_number, chunk in enumerate(chunks, start=1):
            
            embedding = self.get_embedding(chunk['text'])
            point_id = int(uuid.uuid4().int & 0xFFFFFFFF)
            point = PointStruct(
                id=point_id,
                vector=embedding,
                payload={
                    "doc_number": doc_number,
                    "language": "en",
                    "page_content": chunk['text'],
                    "region" : self.region,
                }
            )
            points.append(point)

        # Upsert points in batches
        batch_size = 100
        for i in range(0, len(points), batch_size):
            batch = points[i:i + batch_size]
            self.client.upsert(
                collection_name=self.collection_name,
                points=batch
            )

        print(f"{len(points)} transcript documents added.")

    def process_folders(self):
        """
        Process and push data for a list of folders.

        Args:
            folder_list (list): List of folder paths to process.
            reset_collection (bool): Whether to clear and recreate the Qdrant collection before processing.
        """
        if self.reset_collection:
            print(f"Resetting collection '{self.collection_name}'...")
            self.client.delete_collection(self.collection_name)
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
            )
            print(f"Collection '{self.collection_name}' reset successfully.")
        else:
            print(f"Appending to existing collection '{self.collection_name}'...")

        #run push_transcriopts for only one file
        self.push_transcripts()

    def close(self):
        """Close Qdrant client."""
        self.client.close()
        print("Qdrant client closed.")

In [12]:
video_processor = VideoProcessorQdrant(
    collection_name="test_collection",
    file_path="cleaned_text_chunks.json",
    embedding_model="text-embedding-3-large",
    reset_collection=False,
    openai_api_key=openai_api_key,
    region = "India",)

Using existing collection 'test_collection'.


In [13]:
process_folders = video_processor.process_folders()

Appending to existing collection 'test_collection'...
170 transcript documents added.


In [14]:
import json
def load_chunks(file_path):
        """Load chunks from a file."""
        with open(file_path, "r") as f:
            data = json.load(f)
        return data

In [15]:
file_path = "qa_100_responses.json"
data= load_chunks(file_path)

In [16]:
data[0]['qa_pairs']['qa_pairs'][0]['question']

'young leaves look funny, what wrong?'

In [17]:
questions = []

for item in data:
    qa_pairs = item['qa_pairs']['qa_pairs']
    for qa in qa_pairs:
        questions.append(qa['question'])

print(f"Total questions collected: {len(questions)}")


Total questions collected: 496


In [18]:
questions[0]

'young leaves look funny, what wrong?'

In [20]:
class RAGRetriever:
    def __init__(self, collection_name="test_collection", embedding_model="text-embedding-3-large"):
        self.client = QdrantClient(url="http://dev.platform.farmer.chat:5438/", port=5438, grpc_port=5439, prefer_grpc=False)
        self.collection_name = collection_name
        self.embedding_model = embedding_model
        self.openai_client = OpenAI(api_key=openai_api_key)

    def get_embedding(self, text):
        """Get embedding for text using OpenAI."""
        response = self.openai_client.embeddings.create(
            input=text,
            model=self.embedding_model
        )
        return response.data[0].embedding

    def retrieve(self, query, top_k=5):
        """
        Retrieve relevant documents for a query.
        
        Args:
            query (str): The search query
            top_k (int): Number of results to return
        
        Returns:
            list: Retrieved documents with their metadata
        """
        # Get embedding for the query
        query_vector = self.get_embedding(query)
        
        # Search in Qdrant
        search_results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=top_k
        )
        
        # Format results
        results = []
        for result in search_results:
            results.append({
                'content': result.payload['page_content'],
                'region': result.payload['region'],
                'score': result.score,
                'doc_number': result.payload['doc_number']
            })
        
        return results
    def get_llm_response(self, query, retrieved_docs, temperature=0.0):
        """
        Generate LLM response based on the query and retrieved documents.
        
        Args:
            query (str): User's question
            retrieved_docs (list): List of retrieved documents
            temperature (float): Temperature for LLM response generation
            
        Returns:
            str: LLM generated response
        """
        # Construct the context from retrieved documents
        context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" 
                            for i, doc in enumerate(retrieved_docs)])
        
        # Construct the prompt
        prompt = f"""You are Farmer.CHAT, a highly knowledgeable AI assistant specializing in farming, aimed at providing comprehensive assistance in various farming practices.
                    You are chatting with the farmer, who is a person in the farming community. Your task is to be friendly and ask the user politly for the information required to answer. 
                    Whenever possible, try to format your answers using bullet points and new lines to improve readability. 
                    Try to decorate your answers with emojis compatible with telegram messenger.
        Context:
        {context}
        
        Question: {query}
        
        Answer:
        """
        
        # Get response from GPT
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that provides accurate answers based on the given context."},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
        )
        
        return response.choices[0].message.content

    def retrieve_and_generate(self, query, top_k=3):
        """
        Combined method to retrieve documents and generate LLM response.
        
        Args:
            query (str): User's question
            top_k (int): Number of documents to retrieve
            
        Returns:
            dict: Dictionary containing retrieved documents and LLM response
        """
        # Retrieve relevant documents
        retrieved_docs = self.retrieve(query, top_k=top_k)
        
        # Generate LLM response
        llm_response = self.get_llm_response(query, retrieved_docs)
        
        return {
            'query': query,
            'retrieved_documents': retrieved_docs,
            'llm_response': llm_response
        }
    def process_queries(self, questions, top_k=3):
        """
        Process multiple queries and get their relevant documents.
        
        Args:
            questions (list): List of questions to process
            top_k (int): Number of results to return per query
            
        Returns:
            dict: Questions mapped to their retrieved contexts
        """
        results = {}
        for i, question in enumerate(questions):
            retrieved_docs = self.retrieve(question, top_k=top_k)
            results[question] = retrieved_docs
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1} queries...")
        
        return results

In [21]:
# Initialize the RAG retriever
retriever = RAGRetriever()

# Test with a single query
query = questions[0]
result = retriever.retrieve_and_generate(query, top_k=3)

# Print results
print(f"Query: {result['query']}\n")
print("LLM Response:")
print(result['llm_response'])
print("\nRetrieved Documents:")
for i, doc in enumerate(result['retrieved_documents'], 1):
    print(f"\n{i}. Content: {doc['content'][:200]}...")
    print(f"   Score: {doc['score']:.4f}")
    print(f"   Region: {doc['region']}")



Query: young leaves look funny, what wrong?

LLM Response:
🌱 Hello there! It sounds like your young leaves are showing some unusual symptoms. To help you better, could you please provide a bit more information? Here are a few questions:

- What type of plants are you growing? 🌾
- Are the leaves distorted, small, or dark green? 🤔
- Do you notice any curling, crinkling, or discoloration? 🌈
- Have you observed any other symptoms, like premature shedding of buds or flowers? 🌼

With this information, I can give you a more accurate assessment of what might be going wrong! 😊

Retrieved Documents:

1. Content: 2. The young leaves of new plants are affected first. These are often distorted, small and ab-normally dark green. 3. Leaves may be cup-shaped and crinkled and the terminal buds deteriorate with some ...
   Score: 0.4542
   Region: India

2. Content: 2. In maize, from light yellow striping to a broad band of white or yellow tissue with reddish purple veins between the midrib and edges of

In [22]:
result

{'query': 'young leaves look funny, what wrong?',
 'retrieved_documents': [{'content': '2. The young leaves of new plants are affected first. These are often distorted, small and ab-normally dark green. 3. Leaves may be cup-shaped and crinkled and the terminal buds deteriorate with some breakdown of petioles. 4. Root growth is markedly impaired; rooting of roots occurs. 5. Dessication of growing points (terminal buds) of plants under severe deficiency. 6. Buds and blossoms shed prematurely. 7. Stem structure weakened. Magnesium (Mg) deficiency symptoms 1. Interveinal chlorosis, mainly of older leaves, producing a streaked or patchy effect; with acute deficiency, the affected tissue may dry up and die. 2. Leaves usually small, brittle in final stages and curve upwards at margin. 3. In some vegetable plants, chlorotic spot be-tween veins, with tints of orange, red and purple. 4. Twigs weak and prone to fungus attack, usu-ally premature, leaf drop. Sulphur (S) deficiency symptoms 1. Y oun

In [ ]:
# To process multiple questions:
def process_multiple_queries(questions, batch_size=5):
    results = []
    for i, question in enumerate(questions[:batch_size]):  # Process first batch_size questions
        result = retriever.retrieve_and_generate(question)
        results.append(result)
        print(f"Processed query {i+1}/{batch_size}")
    return results